# Notebook 45 — Does the onset scale with filter count?

The onset of collapse at roughly 100 to 200 surviving input weights was measured on one first-layer geometry
(64 filters, kernel 3, 192 weights). This notebook trains the same CNN1D with 32 filters (96 first-layer weights) and
128 filters (384 weights), five seeds each, and repeats the conv.0 dose sweep with the rest of the network at 80%.
Doses are chosen so that the surviving-weight counts overlap across geometries (including 96, 38 and 19), which
separates two readings of the onset: an absolute count of surviving weights or live filters, versus a fraction of the
layer.

**Gate (stated before running).** The onset is absolute if, at 38 surviving first-layer weights, both geometries
collapse (mean macro-F1 loss > 0.15) and their losses at matched surviving counts agree within 0.10; it is fractional
if the 128-filter network at 38 weights (10% of its layer) is damaged materially more than the 32-filter network at
38 weights (40% of its layer). Baselines trained here are validation-gated against the 64-filter band. Resumable per
(geometry, seed, dose). GPU runtime required.

In [ ]:
# --- Colab bootstrap ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys, copy, json as _json
os.chdir(REPO); sys.path.insert(0, REPO)
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.utils.prune as prune
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from src.config import CFG, PATHS, set_all_seeds
from src.data import load_raw, clean, temporal_within_capture_split
from src import train as TR, models as M, explain as EXP, mitigate
from src.comnet_audit import assign_validation_tiers, calibration_summary, environment_record, write_json
from src.train import load_anchor, predict, per_class_recall_table, feature_columns

assert torch.cuda.is_available(), 'switch to a GPU runtime first'
DEVICE = TR.DEVICE
DATASET = 'ciciot2023'
SEEDS = list(CFG['seeds']); ANCHOR = int(CFG['anchor_seed'])
OUT = PATHS.tables('comnet')
PRACTICAL_LOSS = 0.10
GEOMS = {32: {'channels': (32, 128)}, 128: {'channels': (128, 128)}}            # first-layer filters -> arch kwargs
DOSES = {32: [0.0, 0.5, 0.604, 0.802, 0.896], 128: [0.0, 0.5, 0.75, 0.901, 0.951]}   # -> 96/48/38/19/10 and 384/192/96/38/19 surviving weights

ARCH = 'cnn1d'
from src.train import train_model
from scipy.stats import spearmanr
for F, ds in DOSES.items(): print(f'{F} filters ({F*3} weights): surviving ->', [int(round(F*3*(1-d))) for d in ds])

In [ ]:
df = clean(load_raw(DATASET, subsample=True, seed=ANCHOR), DATASET)
splits = temporal_within_capture_split(df, seed=ANCHOR)
feat_cols = feature_columns(df)
print(f'{len(df):,} rows | {df.label.nunique()} classes')

In [ ]:
# Pruning policies on the CNN. Fine-tune loop identical to src.compression.prune_and_finetune.
def prunable(model):
    return [(mod, 'weight') for mod in model.modules() if isinstance(mod, (nn.Linear, nn.Conv1d))]

def layer_names(model):
    return {mod: n for n, mod in model.named_modules()}

def apply_layer_amounts(model, amounts):
    # amounts: {module_name: sparsity}; every prunable layer must be named (no silent defaults)
    m = copy.deepcopy(model); names = layer_names(m)
    for mod, name in prunable(m):
        a = amounts[names[mod]]
        if a > 0: prune.l1_unstructured(mod, name=name, amount=float(a)); prune.remove(mod, name)
    return m

def layer_sparsity(model):
    names = layer_names(model); out = {}
    for mod, name in prunable(model):
        w = getattr(mod, name); out[names[mod]] = float((w == 0).float().mean())
    z = sum(int((getattr(mod, n) == 0).sum()) for mod, n in prunable(model)); n_ = sum(getattr(mod, n).numel() for mod, n in prunable(model))
    out['prunable_sparsity'] = z / n_; out['remaining_nonzero_prunable'] = n_ - z
    return out

def finetune_masked(model, seed, *, ft_epochs=8, batch_size=4096, lr=5e-4, verbose=False):
    set_all_seeds(seed)
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    le = LabelEncoder().fit(df['label'].to_numpy())
    scaler = StandardScaler().fit(df.loc[splits['train'], feat_cols].to_numpy(np.float32))
    t = TR.make_tensors(df, splits, feat_cols, le, scaler); Xtr, ytr = t['train']
    model = model.to(DEVICE)
    masks = {(mod, name): (getattr(mod, name) != 0).float() for mod, name in prunable(model)}
    hooks = [getattr(mod, name).register_hook((lambda mk: (lambda g: g * mk))(mk)) for (mod, name), mk in masks.items()]
    w = TR.tempered_class_weights(ytr.numpy(), len(le.classes_))
    crit = nn.CrossEntropyLoss(weight=w); opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    for ep in range(ft_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        if verbose: print(f'    ft epoch {ep}')
    for h in hooks: h.remove()
    with torch.no_grad():
        for mod, name in prunable(model): getattr(mod, name).mul_((getattr(mod, name) != 0).float())
    return model.eval(), le, scaler

def save_ckpt(model, le, scaler, path):
    torch.save({'state_dict': model.state_dict(), 'classes': list(le.classes_), 'feat_cols': feat_cols,
                'scaler_mean': scaler.mean_, 'scaler_scale': scaler.scale_}, path)

def get_baseline(F, seed):
    cell = f'M0_f{F}_paired'; p = PATHS.model(DATASET, ARCH, cell, seed)
    if os.path.exists(p): return load_anchor(DATASET, ARCH, cell, seed, arch_kwargs=GEOMS[F])
    print(f'  training {F}-filter baseline seed {seed}')
    m, info = train_model(ARCH, df, DATASET, splits, seed, epochs=40, patience=6, batch_size=4096, lr=1e-3, compression=cell, arch_kwargs=GEOMS[F], save=True, verbose=False)
    return m, info['label_encoder'], info['scaler'], feat_cols
def live_filters(model):
    names = layer_names(model); conv0 = [mod for mod, _ in prunable(model) if names[mod] == 'conv.0'][0]
    nz = (conv0.weight.detach() != 0).reshape(conv0.weight.shape[0], -1); return int(nz.any(dim=1).sum()), int(nz.sum())
for F in GEOMS:
    m_, _, _, _ = get_baseline(F, ANCHOR); got = [layer_names(m_)[mod] for mod, _ in prunable(m_)]; assert got == ['conv.0', 'conv.3', 'head'], got
    w = [mod for mod, _ in prunable(m_) if layer_names(m_)[mod] == 'conv.0'][0].weight; assert w.numel() == F * 3, (F, w.numel()); print(f'  {F} filters: conv.0 has {w.numel()} weights')
# validation gate against the archived 64-filter CNN band
bands = pd.read_csv(OUT / 'validation_architecture_gate.csv'); cnn_band = (float(bands[bands.arch == 'cnn1d'].validation_macro_f1.min()), float(bands[bands.arch == 'cnn1d'].validation_macro_f1.max()))
print('64-filter validation band:', cnn_band)

In [ ]:
# Baselines (validation gate) and dose sweep, with resume
baseline_val, baseline_test, comp_test, macro, ls_rows, gate_rows = {}, {}, {}, [], [], []
for F, ds in DOSES.items():
    for seed in SEEDS:
        print(f'\n===== {F} filters, seed {seed} =====')
        m0, le, scaler, _ = get_baseline(F, seed)
        yv, pv, _ = predict(m0, df, splits, le, scaler, feat_cols, which='val'); yt, pt, _ = predict(m0, df, splits, le, scaler, feat_cols, which='test')
        baseline_val[(F, seed)] = per_class_recall_table(yv, pv, le).set_index('label')['recall']; baseline_test[(F, seed)] = per_class_recall_table(yt, pt, le).set_index('label')['recall']
        vf1 = f1_score(yv, pv, average='macro'); gate_rows.append({'filters': F, 'seed': seed, 'validation_macro_f1': vf1})
        macro.append({'filters': F, 'seed': seed, 'dose': -1.0, 'cell': 'M0', 'test_macro_f1': f1_score(yt, pt, average='macro')})
        for d in ds:
            cell = f'f{F}_conv0dose{int(round(d*1000))}_paired'; p_c = PATHS.model(DATASET, ARCH, cell, seed)
            if os.path.exists(p_c):
                mp = M.build(ARCH, len(feat_cols), len(le.classes_), **GEOMS[F]).to(DEVICE); mp.load_state_dict(torch.load(p_c, map_location=DEVICE, weights_only=False)['state_dict']); mp.eval(); print(f'  loaded {cell}')
            else:
                mp, _, _ = finetune_masked(apply_layer_amounts(m0, {'conv.0': d, 'conv.3': 0.8, 'head': 0.8}), seed); save_ckpt(mp, le, scaler, p_c); print(f'  saved {cell}')
            live, taps = live_filters(mp); ls_rows.append({'filters': F, 'seed': seed, 'dose': d, 'cell': cell, 'live_filters': live, 'surviving_input_weights': taps, **layer_sparsity(mp)})
            yt, pc, _ = predict(mp, df, splits, le, scaler, feat_cols, which='test')
            comp_test[(F, seed, d)] = per_class_recall_table(yt, pc, le).set_index('label')['recall']
            macro.append({'filters': F, 'seed': seed, 'dose': d, 'cell': cell, 'test_macro_f1': f1_score(yt, pc, average='macro')})
gdf = pd.DataFrame(gate_rows); gdf.to_csv(OUT / 'filter_sweep_validation_gate.csv', index=False)
for F in GEOMS:
    b = gdf[gdf.filters == F].validation_macro_f1; print(f'{F} filters validation band {b.min():.3f}-{b.max():.3f} | overlaps 64-filter band {cnn_band}:', not (b.max() < cnn_band[0] or b.min() > cnn_band[1]))
print('\nsweep complete')

In [ ]:
# Aggregate + gate
rows = []
for F in GEOMS:
    tiers = assign_validation_tiers(pd.DataFrame({s: baseline_val[(F, s)] for s in SEEDS}))
    for (F_, seed, d), rc in comp_test.items():
        if F_ != F: continue
        r0 = baseline_test[(F, seed)]
        for cls in r0.index.intersection(rc.index):
            loss = float(r0.loc[cls] - rc.loc[cls]); band = float(tiers.loc[cls, 'validation_2sd_band'])
            rows.append({'filters': F, 'seed': seed, 'dose': d, 'class': cls, 'recall_loss': loss, 'material_and_beyond_band': bool((loss > band) and (loss >= PRACTICAL_LOSS))})
eff = pd.DataFrame(rows); assert len(eff) > 0, 'no rows: run the sweep cell in this session first'; eff.to_csv(OUT / 'filter_sweep_per_class_effects.csv', index=False)
summ = eff.groupby(['filters', 'dose', 'class']).agg(mean_recall_loss=('recall_loss', 'mean'), affected_frequency=('material_and_beyond_band', 'mean')).reset_index(); summ.to_csv(OUT / 'filter_sweep_per_class_summary.csv', index=False)
mdf = pd.DataFrame(macro); mdf.to_csv(OUT / 'filter_sweep_macro_f1_wide.csv', index=False); lsdf = pd.DataFrame(ls_rows); lsdf.to_csv(OUT / 'filter_sweep_layer_sparsity.csv', index=False)
tab = []
for F, ds in DOSES.items():
    m0m = mdf[(mdf.filters == F) & (mdf.cell == 'M0')].test_macro_f1.mean()
    for d in ds:
        g = mdf[(mdf.filters == F) & (mdf.dose == d)].test_macro_f1; l = lsdf[(lsdf.filters == F) & (lsdf.dose == d)]
        tab.append({'filters': F, 'first_layer_weights': F * 3, 'conv0_sparsity': d, 'surviving_input_weights': int(round(l.surviving_input_weights.mean())), 'live_filters': float(l.live_filters.mean()),
                    'live_fraction': float(l.live_filters.mean() / F), 'mean_macro_f1': g.mean(), 'sd_macro_f1': g.std(), 'mean_macro_f1_loss': m0m - g.mean(),
                    'classes_affected_ge3of5': int((summ[(summ.filters == F) & (summ.dose == d)].affected_frequency >= 0.6).sum())})
ref = pd.read_csv(OUT / 'input_starvation_dose_response.csv'); ref = ref[ref.arch == 'cnn1d']
for _, r in ref.iterrows():
    tab.append({'filters': 64, 'first_layer_weights': 192, 'conv0_sparsity': r.input_layer_sparsity, 'surviving_input_weights': int(r.surviving_input_weights), 'live_filters': np.nan, 'live_fraction': np.nan,
                'mean_macro_f1': r.mean_macro_f1, 'sd_macro_f1': r.sd_macro_f1, 'mean_macro_f1_loss': r.mean_macro_f1_loss, 'classes_affected_ge3of5': int(r.classes_affected_ge3of5)})
tab = pd.DataFrame(tab).sort_values(['filters', 'surviving_input_weights'], ascending=[True, False]); tab.to_csv(OUT / 'filter_sweep_dose_response.csv', index=False)
pd.set_option('display.width', 220); print(tab.round(4).to_string(index=False))
def at(F, w):
    t = tab[(tab.filters == F)]; return t.iloc[(t.surviving_input_weights - w).abs().argsort().iloc[0]]
l32, l128, l64 = at(32, 38), at(128, 38), at(64, 38)
verdict = pd.DataFrame([
 {'criterion': 'both_new_geometries_collapse_at_38_weights_loss_gt_0.15', 'value': f'32f {l32.mean_macro_f1_loss:.3f}, 128f {l128.mean_macro_f1_loss:.3f}', 'pass': bool(l32.mean_macro_f1_loss > 0.15 and l128.mean_macro_f1_loss > 0.15)},
 {'criterion': 'losses_at_38_weights_agree_within_0.10_across_32_64_128', 'value': f'range {max(l32.mean_macro_f1_loss, l64.mean_macro_f1_loss, l128.mean_macro_f1_loss) - min(l32.mean_macro_f1_loss, l64.mean_macro_f1_loss, l128.mean_macro_f1_loss):.3f}', 'pass': bool(max(l32.mean_macro_f1_loss, l64.mean_macro_f1_loss, l128.mean_macro_f1_loss) - min(l32.mean_macro_f1_loss, l64.mean_macro_f1_loss, l128.mean_macro_f1_loss) <= 0.10)},
 {'criterion': 'fractional_reading_128f_at_38_worse_than_32f_at_38_by_gt_0.10', 'value': round(float(l128.mean_macro_f1_loss - l32.mean_macro_f1_loss), 3), 'pass': bool(l128.mean_macro_f1_loss - l32.mean_macro_f1_loss > 0.10)},
 {'criterion': 'ref_spearman_surviving_weights_vs_loss_all_geometries', 'value': round(float(spearmanr(tab.surviving_input_weights, tab.mean_macro_f1_loss).correlation), 3), 'pass': ''},
])
print(); print(verdict.to_string(index=False)); ab = bool(verdict['pass'].iloc[0] and verdict['pass'].iloc[1]); fr = bool(verdict['pass'].iloc[2])
print('\nOnset reads as an ABSOLUTE count of surviving first-layer weights:', ab, '| as a FRACTION of the layer:', fr)
verdict.to_csv(OUT / 'filter_sweep_gate_verdict.csv', index=False)
write_json(OUT / 'filter_sweep_environment.json', {'geometries': {str(k): v for k, v in GEOMS.items()}, 'doses': {str(k): v for k, v in DOSES.items()}, 'seeds': SEEDS, 'environment': environment_record()})

In [ ]:
# --- Commit + push: main only, this notebook's own files only ---
import subprocess, shutil, glob
_b = subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert _b == 'main', f'checked-out branch is {_b!r}; run `git checkout main` first'
subprocess.run(['git', 'config', '--global', 'user.name', 'Md Anas Biswas'], check=True)
subprocess.run(['git', 'config', '--global', 'user.email', 'anasbiswas@gmail.com'], check=True)
cred = '/content/drive/MyDrive/IoT_Trust_Research/.git-credentials'
if os.path.exists(cred):
    shutil.copy(cred, '/root/.git-credentials'); subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
_own = 'notebooks/45_filter_count_sweep.ipynb'
if os.path.exists(_own):
    d = _json.load(open(_own))
    for c in d.get('cells', []):
        if c.get('cell_type') == 'code': c['outputs'] = []; c['execution_count'] = None
    _json.dump(d, open(_own, 'w'), indent=1)
subprocess.run(['git', 'add', _own] + glob.glob('results/tables/comnet/filter_sweep_*'), check=True)
r = subprocess.run(['git', 'commit', '-m', 'notebook 45: filter-count sweep (32 and 128 filters, five seeds each) - is the collapse onset an absolute surviving-weight count or a fraction of the layer'], capture_output=True, text=True)
print(r.stdout or r.stderr)
print(subprocess.run(['git', 'push'], capture_output=True, text=True).stderr or 'pushed')
print(subprocess.run(['git', 'log', '--oneline', '-2'], capture_output=True, text=True).stdout)